# Zadanie - czytanie/zapisywanie i pochodny typ zmiennej
Karta pracy do samodzielnego rozwiązania.

Cel:
- połączyć pracę z plikiem, moduł i typ pochodny,
- wykonać konwersje jednostek,
- zapisać wynik do nowego, czytelnego pliku.


## 1) Treść zadania

Napisz program, który:
1. wczyta z pliku `stars.dat` dane gwiazd,
2. zapisze dane do nowego pliku w przekształconej formie,
3. policzy jasność `L` jako funkcję,
4. wszystkie funkcje i stałe umieści w module.

Kolumny wejściowe (`stars.dat`):
- `Nazwa`, `Ra [deg]`, `Dec [deg]`, `Masa [M_sun]`, `Promien [R_sun]`, `T_eff [K]`

Wymagane wyjście:
- `Nazwa` bez zmian
- `Ra`: `deg -> HH:MM:SS`
- `Dec`: `deg -> DD:MM:SS`
- `Masa`: `M_sun -> kg`
- `Promien`: `R_sun -> km`
- `T_eff` bez zmian
- `L` w `L_sun`


## 2) Sugerowany podział na moduły/procedury

`module star_tools`
- stałe fizyczne (`M_sun`, `R_sun`, `sigma`, `L_sun`),
- typ pochodny `star`,
- funkcje konwersji kątów (`deg_to_hms`, `deg_to_dms`),
- funkcja `luminosity_lsun(radius_sun, teff)`.

`program main`
- odczyt pliku,
- przetworzenie rekordów,
- zapis raportu.


## 3) Mini-checklista przed oddaniem

- [ ] jest zdefiniowany typ `star`
- [ ] wszystkie stałe/funkcje są w module
- [ ] konwersje RA/Dec dają czytelny format tekstowy
- [ ] plik wynikowy ma nagłówek i stałe formatowanie kolumn
- [ ] `L` jest liczone funkcją, nie „wklejone” w programie głównym


In [ ]:
module star_tools
  implicit none

  real, parameter :: m_sun_kg = 1.98847e30
  real, parameter :: r_sun_km = 695700.0
  real, parameter :: sigma = 5.670374419e-8
  real, parameter :: l_sun_w = 3.828e26

  type :: star
     character(len=30) :: name
     real :: ra_deg
     real :: dec_deg
     real :: mass_sun
     real :: radius_sun
     real :: teff
  end type star

contains

  real function luminosity_lsun(radius_sun, teff)
    implicit none
    real, intent(in) :: radius_sun, teff
    real :: radius_m

    radius_m = radius_sun * r_sun_km * 1000.0
    luminosity_lsun = (4.0 * 3.1415926536 * radius_m**2 * sigma * teff**4) / l_sun_w
  end function luminosity_lsun

  subroutine deg_to_hms(ra_deg, h, m, s)
    implicit none
    real, intent(in) :: ra_deg
    integer, intent(out) :: h, m
    real, intent(out) :: s
    real :: total_hours, frac

    total_hours = ra_deg / 15.0
    h = int(total_hours)
    frac = (total_hours - real(h)) * 60.0
    m = int(frac)
    s = (frac - real(m)) * 60.0
  end subroutine deg_to_hms

  subroutine deg_to_dms(dec_deg, d, m, s)
    implicit none
    real, intent(in) :: dec_deg
    integer, intent(out) :: d, m
    real, intent(out) :: s
    real :: val, frac

    val = abs(dec_deg)
    d = int(val)
    frac = (val - real(d)) * 60.0
    m = int(frac)
    s = (frac - real(m)) * 60.0

    if (dec_deg < 0.0) d = -d
  end subroutine deg_to_dms

end module star_tools

program zadanie_stars
  use star_tools
  implicit none

  type(star), allocatable :: stars(:)
  integer :: n, i, io
  integer :: h, mm, d, dm
  real :: ss, ds, l

  n = 0
  open(10, file='stars.dat', status='old', action='read')
  do
     read(10,*,iostat=io)
     if (io /= 0) exit
     n = n + 1
  end do
  close(10)

  allocate(stars(n))

  open(10, file='stars.dat', status='old', action='read')
  do i = 1, n
     read(10,*) stars(i)%name, stars(i)%ra_deg, stars(i)%dec_deg, &
                stars(i)%mass_sun, stars(i)%radius_sun, stars(i)%teff
  end do
  close(10)

  open(20, file='stars_report.dat', status='replace', action='write')
  write(20,'(a)') "# Raport gwiazd"

  do i = 1, n
     call deg_to_hms(stars(i)%ra_deg, h, mm, ss)
     call deg_to_dms(stars(i)%dec_deg, d, dm, ds)
     l = luminosity_lsun(stars(i)%radius_sun, stars(i)%teff)

     write(20,'(a)') "Nazwa   : "//trim(stars(i)%name)
     write(20,'(a,i2.2,":",i2.2,":",f6.3)') "Ra      : ", h, mm, ss
     write(20,'(a,i3.3,":",i2.2,":",f6.3)') "Dec     : ", d, dm, ds
     write(20,'(a,es12.4,a)') "Masa    : ", stars(i)%mass_sun*m_sun_kg, " kg"
     write(20,'(a,f12.2,a)') "Promien : ", stars(i)%radius_sun*r_sun_km, " km"
     write(20,'(a,f8.1,a)') "T_eff   : ", stars(i)%teff, " K"
     write(20,'(a,f10.4,a)') "L       : ", l, " L_sun"
     write(20,'(a)') ""
  end do

  close(20)
  deallocate(stars)
end program zadanie_stars


## 4) Podpowiedzi

- Wczytywanie nazw z przecinkami najprościej zrobić list-directed `read(10,*) ...`.
- Dla RA pamiętaj o przeliczeniu `deg -> hour`: `hours = deg / 15`.
- Zadbaj o stały format kolumn i separator między rekordami.


## 5) Dla chętnych

1. Posortuj gwiazdy malejąco po `L`.
2. Dodaj kontrolę jakości danych (np. czy masa/promień > 0).
3. Zapisz również wersję CSV obok raportu tekstowego.
